# Task 2 — Community Detection by Label Propagation

**Dataset.** The same but complete LastFM Asia social network.

**Label propagation-based community detection.** Give every node its own community label. Then
repeatedly visit nodes in random order and let each one adopt the label that is most
popular among its neighbours. The whole algorithm is a loop around a single
decision — *which label should this node take?* — and that decision is what you write.

**What you implement.** The function `score_labels` in section 3. Everything else is provided,
only modify when necessary. Please refer to Section 5 for the introduction of evaluation metrics and tools for diagnosis.

**Note.** Run the cells in order. Please follow the instructions provided in the comments. You are encouraged to use coding agents. 

## 1. Setup

On Google Colab this downloads the data. Running from a local checkout, it finds
the files already there and downloads nothing.


In [ ]:
import json
import random
import urllib.request
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_mutual_info_score

GITHUB_REPO = "antman9914/CSE60745-Practice"
BRANCH = "main"

DATA_DIR = Path("data/task2")
REQUIRED = ["edges_all.csv", "node_country.csv"]

MAX_ITER = 50   # give up after this many sweeps if the labels have not settled
N_SEEDS = 10    # independent runs used for the stability metric in section 5

DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in REQUIRED:
    if not (DATA_DIR / name).exists():
        url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{BRANCH}/data/task2/{name}"
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, DATA_DIR / name)

missing = [n for n in REQUIRED if not (DATA_DIR / n).exists()]
assert not missing, f"could not obtain: {missing}"
assert tuple(map(int, nx.__version__.split(".")[:2])) >= (3, 0), \
    f"this notebook needs networkx >= 3.0, found {nx.__version__}"
print(f"data ready in {DATA_DIR}/ | networkx {nx.__version__}, pandas {pd.__version__}")

pd.set_option("display.width", 120)

## 2. Data loading

In [ ]:
def load_task2_data(data_dir=DATA_DIR):
    """Load the complete LastFM Asia graph and the country label of every node.

    The country label plays no part in community detection. It is loaded here, with
    everything else, only so that it is available for the discussion in section 6.

    Returns
    -------
    G : networkx.Graph
        All 7,624 nodes and all 27,806 edges.
    country : pandas.Series
        node_id -> country, 18 classes. Its index lists every user, including any with
        no edges at all, so it also fixes the node set of the graph.
    """
    edges = pd.read_csv(data_dir / "edges_all.csv")
    country = pd.read_csv(data_dir / "node_country.csv").set_index("node_id")["country"]

    G = nx.Graph()
    G.add_nodes_from(range(int(country.index.max()) + 1))
    G.add_edges_from(edges[["src", "dst"]].itertuples(index=False, name=None))
    return G, country


G, country = load_task2_data()
degrees = np.array([d for _, d in G.degree()])
print(f"nodes {G.number_of_nodes()}, edges {G.number_of_edges()}")
print(f"mean degree {degrees.mean():.2f}, median {np.median(degrees):.0f}, "
      f"max {degrees.max()}")
print(f"connected: {nx.is_connected(G)}, transitivity {nx.transitivity(G):.4f}")
print(f"country: {country.nunique()} classes over {len(country)} nodes")


## 3. Your implementation

`score_labels` is called once per node per sweep. It receives the node, its neighbours,
and the current label of every node in the graph, and must return a score for each
community label the node could adopt. The framework then assigns the highest-scoring
label.

In [ ]:
def score_labels(node, neighbors, labels, G):
    """Score every community label that `node` could adopt.

    Parameters
    ----------
    node : int
        The node currently being updated.
    neighbors : list of int
        Its neighbours in G. Never empty; isolated nodes are skipped by the framework.
    labels : dict[int, int]
        Every node's CURRENT community label. A community label is just an int: at
        initialisation each node is given its own node id as its label, and after that a
        node only ever adopts a label that some other node already carries. So `labels[v]`
        is the community that neighbour `v` belongs to right now. Updates are
        asynchronous, so this already reflects changes made earlier in this same sweep.
        Do not modify it.
    G : networkx.Graph
        The full graph, in case you need degrees or higher-order structural information.

    Returns
    -------
    dict[int, float]
        Maps a candidate community label to its score. The keys must be labels you read
        out of `labels`. The dict must not be empty.

    Notes
    -----
    This function is called about 7,624 times per sweep, so try to keep it cheap.
    """
    raise NotImplementedError("TODO: implement score_labels")

## 4. The label propagation loop

In [ ]:
def run_label_propagation(G, score_fn, seed=0, max_iter=MAX_ITER, verbose=False):
    """Run asynchronous label propagation using `score_fn` as the update rule.

    Initialisation gives every node a unique label. Each sweep visits all nodes in a
    fresh random order and updates them in place, so a node sees the labels that its
    earlier neighbours already adopted during the same sweep. The run stops as soon as a
    full sweep changes nothing, or after `max_iter` sweeps.

    Returns
    -------
    labels : dict[int, int]
    n_iter : int
    converged : bool
    """
    rng = random.Random(seed)
    adj = {u: list(G.neighbors(u)) for u in G}
    labels = {u: u for u in G}
    order = list(G)

    for it in range(1, max_iter + 1):
        rng.shuffle(order)
        n_changed = 0

        for u in order:
            nbrs = adj[u]
            if not nbrs:
                continue                        # an isolated node keeps its own label

            scores = score_fn(u, nbrs, labels, G)
            if not scores:
                continue

            best = max(scores.values())
            winners = [lab for lab, s in scores.items() if s == best]

            # Standard tie-breaking: prefer the label we already have.
            if labels[u] in winners:
                continue
            labels[u] = rng.choice(winners)
            n_changed += 1

        if verbose:
            print(f"  sweep {it:>3}: {n_changed:>5} nodes changed label")
        if n_changed == 0:
            return labels, it, True

    return labels, max_iter, False


def labels_to_communities(labels):
    """Group a node -> label mapping into a list of node sets, largest first."""
    comms = defaultdict(set)
    for node, lab in labels.items():
        comms[lab].add(node)
    return sorted(comms.values(), key=len, reverse=True)

In [ ]:
labels, n_iter, converged = run_label_propagation(G, score_labels, seed=0, verbose=True)
communities = labels_to_communities(labels)
print(f"\nfinished after {n_iter} sweeps (converged={converged}), "
      f"{len(communities)} communities")

## 5. Evaluation Toolset

**Quality**

- `modularity` — how many more edges fall inside communities than you would expect in a
  random graph with the same degree sequence. Higher is better.
- `mean_conductance` — for one community, the fraction of its edge endpoints that leave
  it, averaged over communities and weighted by community size. Lower is better. Where
  modularity looks at internal density, conductance looks at the boundary.

**Stability**

- `stability_across_seeds` runs label propagation from `N_SEEDS` seeds and reports the
  Adjusted Mutual Information (AMI) between every pair of runs. AMI measures how much two
  partitions agree, corrected for the agreement you would expect by chance: 0 for
  unrelated partitions, 1 for identical ones. Label propagation is randomised in both the
  node visit order and the tie-breaking, so this tells you how stable your result is.

**Diagnostics** — help you analyze your experimental results.

- `coverage` — the fraction of all edges that fall inside a community. One community
  containing the whole graph scores a perfect 1.0, so read it next to the sizes below.
- `n_communities`, `largest_share`, `n_singletons`, `size_median`, `size_p90`, and the
  size plot. Label propagation has two classic failure modes and they are opposites:
  collapsing into one giant community that swallows most of the graph, and shattering
  into hundreds of tiny fragments. The size distribution is what tells them apart.

In [ ]:
def community_metrics(G, labels):
    """Quality and diagnostic measures for one partition, computed without ground truth."""
    communities = labels_to_communities(labels)
    sizes = np.array([len(c) for c in communities])
    n_nodes = G.number_of_nodes()

    # Boundary quality: conductance of each community against the rest of the graph,
    # averaged with community size as the weight. A community covering the whole graph
    # has no boundary at all and is skipped.
    conds, weights = [], []
    for c in communities:
        if len(c) == n_nodes:
            continue
        conds.append(nx.conductance(G, c))
        weights.append(len(c))
    mean_conductance = float(np.average(conds, weights=weights)) if conds else float("nan")

    coverage, _ = nx.community.partition_quality(G, communities)

    return {
        "modularity": round(nx.community.modularity(G, communities), 4),
        "mean_conductance": round(mean_conductance, 4),
        "coverage": round(coverage, 4),
        "n_communities": len(communities),
        "largest_share": round(float(sizes.max() / n_nodes), 4),
        "n_singletons": int((sizes == 1).sum()),
        "size_median": int(np.median(sizes)),
        "size_p90": int(np.quantile(sizes, 0.9)),
    }


metrics = community_metrics(G, labels)
for key, value in metrics.items():
    print(f"{key:>18}: {value}")

In [ ]:
def plot_size_distribution(labels, ax=None):
    """Community sizes, largest first, on a log scale."""
    sizes = sorted((len(c) for c in labels_to_communities(labels)), reverse=True)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(range(len(sizes)), sizes, width=1.0)
    ax.set_yscale("log")
    ax.set_xlabel("community (sorted by size)")
    ax.set_ylabel("number of nodes")
    ax.set_title(f"{len(sizes)} communities, largest = {sizes[0]} nodes")
    return ax


plot_size_distribution(labels)
plt.tight_layout()
plt.show()

In [ ]:
def stability_across_seeds(G, score_fn, n_seeds=N_SEEDS, max_iter=MAX_ITER):
    """Run label propagation from several seeds and measure how much the runs agree.

    Returns the per-run metrics and the pairwise AMI between every pair of runs.
    """
    nodes = list(G)
    runs, rows = [], []
    for seed in range(n_seeds):
        lab, n_iter, converged = run_label_propagation(
            G, score_fn, seed=seed, max_iter=max_iter
        )
        runs.append(np.array([lab[u] for u in nodes]))
        rows.append({"seed": seed, "n_iter": n_iter, "converged": converged,
                     **community_metrics(G, lab)})

    amis = [adjusted_mutual_info_score(runs[i], runs[j])
            for i in range(n_seeds) for j in range(i + 1, n_seeds)]
    return pd.DataFrame(rows), np.array(amis)


per_run, amis = stability_across_seeds(G, score_labels)
print(f"pairwise AMI over {len(amis)} pairs: {amis.mean():.3f} +/- {amis.std():.3f} "
      f"(min {amis.min():.3f}, max {amis.max():.3f})")
per_run

## 6. Discussion: what do the communities correspond to?

Last.FM Asia contains node attributes we have not used, one of which is the user's
country, an 18-class label. It is **not** a ground-truth community label — people in the
same country do not necessarily form one community.

Still, it is a valuable external description of the nodes. Design the comparison between
the country distribution and your community partition yourself: pick appropriate
measures, and discuss what you think the relationship is between geographical association
and the quality of a community detector.


In [ ]:
print(country.value_counts())

# Your analysis goes here.
